In [42]:
import os

print("SOURCE:", bool(os.environ.get("SOURCE_SAS_URL")))
print("LAKE:", bool(os.environ.get("LAKE_SAS_URL")))

SOURCE: True
LAKE: True


In [43]:
from azure.storage.blob import ContainerClient
import os

source = ContainerClient.from_container_url(os.environ["SOURCE_SAS_URL"])
lake = ContainerClient.from_container_url(os.environ["LAKE_SAS_URL"])

print("Підключення до Azure успішне!")

Підключення до Azure успішне!


In [44]:
from datetime import datetime, timezone

STUDENT = "yarmolenko_iryna"
LOAD_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

files = ["stations.json", "sensors.json", "readings.json"]

print("Файли у контейнері data-lake:\n")

for file_name in files:
    blob_client = source.get_blob_client(file_name)
    size = blob_client.get_blob_properties().size
    print(f"Файл: {file_name} | Розмір: {size} байт")

Файли у контейнері data-lake:

Файл: stations.json | Розмір: 996 байт
Файл: sensors.json | Розмір: 833 байт
Файл: readings.json | Розмір: 3233 байт


In [45]:
import json
import pandas as pd

data = {}

files = ["stations.json", "sensors.json", "readings.json"]

for file_name in files:
    blob_client = source.get_blob_client(file_name)
    content = blob_client.download_blob().readall()

    records = json.loads(content)

    data[file_name] = records

    print(f"\nФайл: {file_name}")
    print("Тип даних:", type(records).__name__)

    if isinstance(records, list):
        print("Кількість записів:", len(records))
        display(pd.json_normalize(records).head())
    elif isinstance(records, dict):
        print("Ключі:", list(records.keys()))
        display(pd.json_normalize(records).head())


Файл: stations.json
Тип даних: list
Кількість записів: 7


,station_id,name,city,country,installed_at
0,ST01,Центр,Київ,UA,2024-05-10
1,ST02,Оболонь,Kyiv,Ukraine,2024/06/01
2,ST03,Личаків,Львів,україна,2024.06.15
3,ST04,Stare Miasto,Kraków,PL,10/07/2024
4,ST05,Centrum,Warsaw,Poland,NaN



Файл: sensors.json
Тип даних: list
Кількість записів: 8


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m3
1,SN101,ST01,CO2,ppm
2,SN102,ST02,pm2_5,ug/m3
3,SN103,ST03,PM 2.5,µg/m3
4,SN104,ST03,temperature,°C



Файл: readings.json
Тип даних: list
Кількість записів: 23


,reading_id,sensor_id,value,measured_at,status
0,R0001,SN100,23.4,2026-07-01 08:00,ok
1,R0002,SN100,"31,7",2026-07-01 12:00,OK
2,R0003,SN101,610,01/07/2026 08:00,ok
3,R0004,SN102,18.9,2026.07.01 09:00,ok
4,R0005,SN103,45.2,2026-07-01 10:00,ok


In [46]:
# Створюємо RAW-шар у власній папці
raw_prefix = f"{STUDENT}/raw/{LOAD_DATE}"

files = ["stations.json", "sensors.json", "readings.json"]

for file_name in files:
    # Читаємо оригінальний файл із data-lake
    source_blob = source.get_blob_client(file_name)
    content = source_blob.download_blob().readall()

    # Записуємо копію у власну папку students-homework
    target_path = f"{raw_prefix}/{file_name}"
    target_blob = lake.get_blob_client(target_path)

    target_blob.upload_blob(content, overwrite=True)

    print(f"Збережено: {target_path}")

print("\nRAW-шар успішно створено!")

Збережено: yarmolenko_iryna/raw/2026-11-04/stations.json
Збережено: yarmolenko_iryna/raw/2026-11-04/sensors.json
Збережено: yarmolenko_iryna/raw/2026-11-04/readings.json

RAW-шар успішно створено!


In [47]:
print("Файли у власному RAW-шарі:\n")

for blob in lake.list_blobs(name_starts_with=f"{STUDENT}/raw/{LOAD_DATE}/"):
    print(f"{blob.name} | {blob.size} байт")

Файли у власному RAW-шарі:

yarmolenko_iryna/raw/2026-11-04/readings.json | 3233 байт
yarmolenko_iryna/raw/2026-11-04/sensors.json | 833 байт
yarmolenko_iryna/raw/2026-11-04/stations.json | 996 байт


In [48]:
for file_name, records in data.items():
    print(f"\n{'=' * 50}")
    print(f"ФАЙЛ: {file_name}")

    df = pd.json_normalize(records)

    print("\nКолонки:")
    print(df.columns.tolist())

    print("\nТипи даних:")
    print(df.dtypes)

    print("\nПерші 3 записи:")
    display(df.head(3))


ФАЙЛ: stations.json

Колонки:
['station_id', 'name', 'city', 'country', 'installed_at']

Типи даних:
station_id      str
name            str
city            str
country         str
installed_at    str
dtype: object

Перші 3 записи:


,station_id,name,city,country,installed_at
0,ST01,Центр,Київ,UA,2024-05-10
1,ST02,Оболонь,Kyiv,Ukraine,2024/06/01
2,ST03,Личаків,Львів,україна,2024.06.15



ФАЙЛ: sensors.json

Колонки:
['sensor_id', 'station_id', 'metric', 'unit']

Типи даних:
sensor_id     str
station_id    str
metric        str
unit          str
dtype: object

Перші 3 записи:


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m3
1,SN101,ST01,CO2,ppm
2,SN102,ST02,pm2_5,ug/m3



ФАЙЛ: readings.json

Колонки:
['reading_id', 'sensor_id', 'value', 'measured_at', 'status']

Типи даних:
reading_id     str
sensor_id      str
value          str
measured_at    str
status         str
dtype: object

Перші 3 записи:


,reading_id,sensor_id,value,measured_at,status
0,R0001,SN100,23.4,2026-07-01 08:00,ok
1,R0002,SN100,"31,7",2026-07-01 12:00,OK
2,R0003,SN101,610,01/07/2026 08:00,ok


In [49]:
# Перетворюємо JSON у DataFrame
stations_df = pd.json_normalize(data["stations.json"])
sensors_df = pd.json_normalize(data["sensors.json"])
readings_df = pd.json_normalize(data["readings.json"])

# 1. Очищення станцій
stations_df["country"] = (
    stations_df["country"].str.strip().str.lower()
    .replace({
        "ukraine": "UA",
        "україна": "UA",
        "ua": "UA"
    })
)

stations_df["city"] = (
    stations_df["city"].str.strip()
    .replace({
        "Київ": "Kyiv",
        "kyiv": "Kyiv"
    })
)

stations_df["installed_at"] = pd.to_datetime(
    stations_df["installed_at"],
    format="mixed",
    errors="coerce"
)

# 2. Очищення сенсорів
sensors_df["metric"] = (
    sensors_df["metric"].str.strip().str.upper()
    .replace({
        "PM2_5": "PM2.5"
    })
)

sensors_df["unit"] = (
    sensors_df["unit"].str.strip()
    .replace({
        "ug/m3": "µg/m³",
        "µg/m3": "µg/m³"
    })
)

# 3. Очищення показників
readings_df["value"] = pd.to_numeric(
    readings_df["value"].str.replace(",", ".", regex=False),
    errors="coerce"
)

readings_df["measured_at"] = pd.to_datetime(
    readings_df["measured_at"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

readings_df["status"] = (
    readings_df["status"].str.strip().str.lower()
)

print("Очищення завершено!")

display(stations_df.head())
display(sensors_df.head())
display(readings_df.head())

Очищення завершено!


,station_id,name,city,country,installed_at
0,ST01,Центр,Kyiv,UA,2024-05-10
1,ST02,Оболонь,Kyiv,UA,2024-06-01
2,ST03,Личаків,Львів,UA,2024-06-15
3,ST04,Stare Miasto,Kraków,pl,2024-10-07
4,ST05,Centrum,Warsaw,poland,NaT


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m³
1,SN101,ST01,CO2,ppm
2,SN102,ST02,PM2.5,µg/m³
3,SN103,ST03,PM 2.5,µg/m³
4,SN104,ST03,TEMPERATURE,°C


,reading_id,sensor_id,value,measured_at,status
0,R0001,SN100,23.4,2026-01-07 08:00:00,ok
1,R0002,SN100,31.7,2026-01-07 12:00:00,ok
2,R0003,SN101,610.0,2026-07-01 08:00:00,ok
3,R0004,SN102,18.9,2026-01-07 09:00:00,ok
4,R0005,SN103,45.2,2026-01-07 10:00:00,ok


In [50]:
print("STATIONS")
display(stations_df.dtypes)

print("SENSORS")
display(sensors_df.dtypes)

print("READINGS")
display(readings_df.dtypes)

print("Пропущені значення:")
print("Stations:", stations_df.isna().sum().sum())
print("Sensors:", sensors_df.isna().sum().sum())
print("Readings:", readings_df.isna().sum().sum())

STATIONS


station_id                 str
name                       str
city                       str
country                    str
installed_at    datetime64[us]
dtype: object

SENSORS


sensor_id     str
station_id    str
metric        str
unit          str
dtype: object

READINGS


reading_id                str
sensor_id                 str
value                 float64
measured_at    datetime64[us]
status                    str
dtype: object

Пропущені значення:
Stations: 1
Sensors: 1
Readings: 2


In [51]:
print("=== STATIONS: пропущені значення ===")
display(stations_df[stations_df.isna().any(axis=1)])

print("=== SENSORS: пропущені значення ===")
display(sensors_df[sensors_df.isna().any(axis=1)])

print("=== READINGS: пропущені значення ===")
display(readings_df[readings_df.isna().any(axis=1)])

=== STATIONS: пропущені значення ===


,station_id,name,city,country,installed_at
4,ST05,Centrum,Warsaw,poland,NaT


=== SENSORS: пропущені значення ===


,sensor_id,station_id,metric,unit
7,SN107,ST06,PM2.5,NaN


=== READINGS: пропущені значення ===


,reading_id,sensor_id,value,measured_at,status
10,R0010,SN101,NaN,2026-02-07 12:00:00,calibrating
16,R0016,SN104,29.1,NaT,ok


In [52]:
print("Сенсори без відповідної станції:")
display(
    sensors_df[~sensors_df["station_id"].isin(stations_df["station_id"])]
)

print("Показники без відповідного сенсора:")
display(
    readings_df[~readings_df["sensor_id"].isin(sensors_df["sensor_id"])]
)

Сенсори без відповідної станції:


,sensor_id,station_id,metric,unit


Показники без відповідного сенсора:


,reading_id,sensor_id,value,measured_at,status
14,R0014,SN999,40.0,2026-03-07 10:00:00,ok


In [53]:
# Об'єднуємо показники з сенсорами
mart_df = readings_df.merge(
    sensors_df,
    on="sensor_id",
    how="left",
    indicator=True
)

# Додаємо інформацію про станції
mart_df = mart_df.merge(
    stations_df,
    on="station_id",
    how="left"
)

# Зберігаємо кількість записів до фільтрації
print("Записів до очищення:", len(mart_df))

# Залишаємо тільки валідні вимірювання
mart_df = mart_df[
    (mart_df["_merge"] == "both") &
    (mart_df["value"].notna()) &
    (mart_df["measured_at"].notna()) &
    (mart_df["unit"].notna()) &
    (mart_df["status"] == "ok")
].copy()

# Видаляємо технічну колонку
mart_df = mart_df.drop(columns=["_merge"])

# Сортуємо за датою вимірювання
mart_df = mart_df.sort_values("measured_at")

print("Записів у MART:", len(mart_df))

display(mart_df.head(10))

Записів до очищення: 29
Записів у MART: 24


,reading_id,sensor_id,value,measured_at,status,station_id,metric,unit,name,city,country,installed_at
0,R0001,SN100,23.4,2026-01-07 08:00:00,ok,ST01,PM2.5,µg/m³,Центр,Kyiv,UA,2024-05-10
3,R0004,SN102,18.9,2026-01-07 09:00:00,ok,ST02,PM2.5,µg/m³,Оболонь,Kyiv,UA,2024-06-01
8,R0006,SN104,27.5,2026-01-07 10:00:00,ok,ST03,TEMPERATURE,°C,Личаків,Львів,UA,2024-06-15
7,R0005,SN103,45.2,2026-01-07 10:00:00,ok,ST03,PM 2.5,µg/m³,Личаків,Львів,UA,2024-06-15
6,R0005,SN103,45.2,2026-01-07 10:00:00,ok,ST03,PM 2.5,µg/m³,Личаків,Львів,UA,2024-06-15
9,R0006,SN104,27.5,2026-01-07 10:00:00,ok,ST03,TEMPERATURE,°C,Личаків,Львів,UA,2024-06-15
4,R0005,SN103,45.2,2026-01-07 10:00:00,ok,ST03,PM 2.5,µg/m³,Личаків,Львів,UA,2024-06-15
5,R0005,SN103,45.2,2026-01-07 10:00:00,ok,ST03,PM 2.5,µg/m³,Личаків,Львів,UA,2024-06-15
1,R0002,SN100,31.7,2026-01-07 12:00:00,ok,ST01,PM2.5,µg/m³,Центр,Kyiv,UA,2024-05-10
11,R0008,SN105,52.0,2026-02-07 09:00:00,ok,ST04,PM10,µg/m³,Stare Miasto,Kraków,pl,2024-10-07


In [54]:
print("Кількість рядків:", len(mart_df))
print("Кількість колонок:", len(mart_df.columns))

print("\nПропущені значення:")
print(mart_df.isna().sum())

print("\nСтатистика показників:")
display(
    mart_df.groupby(["city", "metric", "unit"])["value"]
    .agg(["count", "min", "max", "mean"])
    .reset_index()
)

Кількість рядків: 24
Кількість колонок: 12

Пропущені значення:
reading_id      0
sensor_id       0
value           0
measured_at     0
status          0
station_id      0
metric          0
unit            0
name            0
city            0
country         0
installed_at    2
dtype: int64

Статистика показників:


,city,metric,unit,count,min,max,mean
0,Kraków,PM10,µg/m³,3,49.8,55.7,52.500000
1,Kyiv,CO2,ppm,2,610.0,720.0,665.000000
2,Kyiv,PM2.5,µg/m³,7,18.9,5000.0,734.685714
3,Warsaw,HUMIDITY,%,2,64.0,71.0,67.500000
4,Львів,PM 2.5,µg/m³,8,38.6,45.2,42.500000
5,Львів,TEMPERATURE,°C,2,27.5,27.5,27.500000


In [55]:
# Створюємо readings заново з оригінального JSON
readings_df = pd.json_normalize(data["readings.json"])

def parse_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    formats = [
        "%Y-%m-%d %H:%M",
        "%d/%m/%Y %H:%M",
        "%Y.%m.%d %H:%M"
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue

    return pd.NaT


readings_df["measured_at"] = readings_df["measured_at"].apply(parse_date)

readings_df["value"] = pd.to_numeric(
    readings_df["value"].str.replace(",", ".", regex=False),
    errors="coerce"
)

readings_df["status"] = readings_df["status"].str.strip().str.lower()

display(readings_df.head(10))

,reading_id,sensor_id,value,measured_at,status
0,R0001,SN100,23.4,2026-07-01 08:00:00,ok
1,R0002,SN100,31.7,2026-07-01 12:00:00,ok
2,R0003,SN101,610.0,2026-07-01 08:00:00,ok
3,R0004,SN102,18.9,2026-07-01 09:00:00,ok
4,R0005,SN103,45.2,2026-07-01 10:00:00,ok
5,R0005,SN103,45.2,2026-07-01 10:00:00,ok
6,R0006,SN104,27.5,2026-07-01 10:00:00,ok
7,R0007,SN100,-3.0,2026-07-02 08:00:00,error
8,R0008,SN105,52.0,2026-07-02 09:00:00,ok
9,R0009,SN106,64.0,2026-07-02 09:00:00,ok


In [56]:
print("Усього записів:", len(readings_df))

print("\nДублікати за reading_id:")
display(
    readings_df[
        readings_df.duplicated(subset=["reading_id"], keep=False)
    ].sort_values("reading_id")
)

print("\nДублікати sensor_id у довіднику:")
display(
    sensors_df[
        sensors_df.duplicated(subset=["sensor_id"], keep=False)
    ].sort_values("sensor_id")
)

Усього записів: 23

Дублікати за reading_id:


,reading_id,sensor_id,value,measured_at,status
4,R0005,SN103,45.2,2026-07-01 10:00:00,ok
5,R0005,SN103,45.2,2026-07-01 10:00:00,ok



Дублікати sensor_id у довіднику:


,sensor_id,station_id,metric,unit


In [57]:
# Видаляємо точні дублікати за reading_id
readings_df = readings_df.drop_duplicates(
    subset=["reading_id"],
    keep="first"
).copy()

# Уніфікуємо назви показників
sensors_df["metric"] = (
    sensors_df["metric"]
    .str.strip()
    .str.upper()
    .replace({
        "PM 2.5": "PM2.5",
        "PM2_5": "PM2.5"
    })
)

# Перевіряємо результат
print("Кількість записів після видалення дубліката:", len(readings_df))
print("Дублікати reading_id:", readings_df["reading_id"].duplicated().sum())

display(sensors_df)

Кількість записів після видалення дубліката: 22
Дублікати reading_id: 0


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m³
1,SN101,ST01,CO2,ppm
2,SN102,ST02,PM2.5,µg/m³
3,SN103,ST03,PM2.5,µg/m³
4,SN104,ST03,TEMPERATURE,°C
5,SN105,ST04,PM10,µg/m³
6,SN106,ST05,HUMIDITY,%
7,SN107,ST06,PM2.5,NaN


In [58]:
mart_df = readings_df.merge(
    sensors_df,
    on="sensor_id",
    how="inner"
)

mart_df = mart_df.merge(
    stations_df,
    on="station_id",
    how="inner"
)

# Залишаємо тільки валідні вимірювання
mart_df = mart_df[
    (mart_df["value"].notna()) &
    (mart_df["measured_at"].notna()) &
    (mart_df["unit"].notna()) &
    (mart_df["status"] == "ok")
].copy()

# Перевірка унікальності
mart_df = mart_df.drop_duplicates(subset=["reading_id"])

mart_df = mart_df.sort_values("measured_at")

print("Кількість записів у MART:", len(mart_df))
print("Дублікати:", mart_df["reading_id"].duplicated().sum())

display(mart_df)

Кількість записів у MART: 18
Дублікати: 0


,reading_id,sensor_id,value,measured_at,status,station_id,metric,unit,name,city,country,installed_at
0,R0001,SN100,23.4,2026-07-01 08:00:00,ok,ST01,PM2.5,µg/m³,Центр,Kyiv,UA,2024-05-10
2,R0003,SN101,610.0,2026-07-01 08:00:00,ok,ST01,CO2,ppm,Центр,Kyiv,UA,2024-05-10
3,R0004,SN102,18.9,2026-07-01 09:00:00,ok,ST02,PM2.5,µg/m³,Оболонь,Kyiv,UA,2024-06-01
4,R0005,SN103,45.2,2026-07-01 10:00:00,ok,ST03,PM2.5,µg/m³,Личаків,Львів,UA,2024-06-15
6,R0006,SN104,27.5,2026-07-01 10:00:00,ok,ST03,TEMPERATURE,°C,Личаків,Львів,UA,2024-06-15
1,R0002,SN100,31.7,2026-07-01 12:00:00,ok,ST01,PM2.5,µg/m³,Центр,Kyiv,UA,2024-05-10
9,R0008,SN105,52.0,2026-07-02 09:00:00,ok,ST04,PM10,µg/m³,Stare Miasto,Kraków,pl,2024-10-07
10,R0009,SN106,64.0,2026-07-02 09:00:00,ok,ST05,HUMIDITY,%,Centrum,Warsaw,poland,NaT
12,R0011,SN102,22.1,2026-07-02 12:00:00,ok,ST02,PM2.5,µg/m³,Оболонь,Kyiv,UA,2024-06-01
13,R0012,SN100,5000.0,2026-07-03 08:00:00,ok,ST01,PM2.5,µg/m³,Центр,Kyiv,UA,2024-05-10


In [59]:
print("Пропущені значення:")
print(mart_df.isna().sum())

print("\nСтатистика за містами та показниками:")

display(
    mart_df.groupby(["city", "metric", "unit"])["value"]
    .agg(["count", "min", "max", "mean"])
    .reset_index()
)

Пропущені значення:
reading_id      0
sensor_id       0
value           0
measured_at     0
status          0
station_id      0
metric          0
unit            0
name            0
city            0
country         0
installed_at    2
dtype: int64

Статистика за містами та показниками:


,city,metric,unit,count,min,max,mean
0,Kraków,PM10,µg/m³,3,49.8,55.7,52.500000
1,Kyiv,CO2,ppm,2,610.0,720.0,665.000000
2,Kyiv,PM2.5,µg/m³,7,18.9,5000.0,734.685714
3,Warsaw,HUMIDITY,%,2,64.0,71.0,67.500000
4,Львів,PM2.5,µg/m³,3,38.6,45.2,41.600000
5,Львів,TEMPERATURE,°C,1,27.5,27.5,27.500000


In [60]:
display(
    mart_df[
        (mart_df["metric"] == "PM2.5")
        & (mart_df["value"] > 100)
    ][
        ["reading_id", "sensor_id", "city", "value",
         "unit", "measured_at", "status"]
    ]
)

,reading_id,sensor_id,city,value,unit,measured_at,status
13,R0012,SN100,Kyiv,5000.0,µg/m³,2026-07-03 08:00:00,ok


In [61]:
print("Запис у readings:")
display(
    readings_df[readings_df["reading_id"] == "R0012"]
)

print("Характеристики датчика:")
display(
    sensors_df[sensors_df["sensor_id"] == "SN100"]
)

Запис у readings:


,reading_id,sensor_id,value,measured_at,status
12,R0012,SN100,5000.0,2026-07-03 08:00:00,ok


Характеристики датчика:


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m³


In [62]:
from io import BytesIO

In [63]:
buffer = BytesIO()

mart_df.to_parquet(buffer, index=False, engine="pyarrow")
buffer.seek(0)

mart_path = f"{STUDENT}/mart/{LOAD_DATE}/air_quality_mart.parquet"

lake.get_blob_client(mart_path).upload_blob(
    buffer.getvalue(),
    overwrite=True
)

print("MART успішно збережено:")
print(mart_path)
print("Кількість записів:", len(mart_df))

MART успішно збережено:
yarmolenko_iryna/mart/2026-11-04/air_quality_mart.parquet
Кількість записів: 18


In [64]:
mart_blobs = list(
    lake.list_blobs(
        name_starts_with=f"{STUDENT}/mart/{LOAD_DATE}/"
    )
)

for blob in mart_blobs:
    print(blob.name, "|", blob.size, "bytes")

yarmolenko_iryna/mart/2026-11-04/air_quality_mart.parquet | 7522 bytes


In [65]:
for file_name in ["stations.json", "sensors.json", "readings.json"]:
    blob = source.get_blob_client(file_name)
    properties = blob.get_blob_properties()

    print(file_name, "-", properties.size, "bytes")

stations.json - 996 bytes
sensors.json - 833 bytes
readings.json - 3233 bytes


In [66]:
def load_raw():
    tables = ["stations", "sensors", "readings"]

    for table in tables:
        file_name = f"{table}.json"

        # Завантажуємо оригінальний файл як байти
        source_blob = source.get_blob_client(file_name)
        content = source_blob.download_blob().readall()

        # Формуємо шлях у RAW
        raw_path = (
            f"{STUDENT}/raw/{table}/"
            f"load_date={LOAD_DATE}/{table}.json"
        )

        # Записуємо без перетворень із перезаписом
        target_blob = lake.get_blob_client(raw_path)
        target_blob.upload_blob(content, overwrite=True)

    print("RAW успішно завантажено.")

In [67]:
# Перший запуск
load_raw()

tables = ["stations", "sensors", "readings"]

print("\nПеревірка розмірів файлів:")

for table in tables:
    file_name = f"{table}.json"

    source_size = source.get_blob_client(
        file_name
    ).get_blob_properties().size

    raw_path = (
        f"{STUDENT}/raw/{table}/"
        f"load_date={LOAD_DATE}/{table}.json"
    )

    raw_size = lake.get_blob_client(
        raw_path
    ).get_blob_properties().size

    print(
        f"{file_name}: "
        f"джерело = {source_size} байт, "
        f"RAW = {raw_size} байт, "
        f"збігаються = {source_size == raw_size}"
    )

# Кількість блобів після першого запуску
raw_prefix = f"{STUDENT}/raw/"
count_first = sum(
    1 for _ in lake.list_blobs(name_starts_with=raw_prefix)
)

# Другий запуск
load_raw()

# Кількість блобів після другого запуску
count_second = sum(
    1 for _ in lake.list_blobs(name_starts_with=raw_prefix)
)

print("\nПеревірка повторного запуску:")
print("Кількість після першого запуску:", count_first)
print("Кількість після другого запуску:", count_second)
print("Кількість не змінилася:", count_first == count_second)

RAW успішно завантажено.

Перевірка розмірів файлів:
stations.json: джерело = 996 байт, RAW = 996 байт, збігаються = True
sensors.json: джерело = 833 байт, RAW = 833 байт, збігаються = True
readings.json: джерело = 3233 байт, RAW = 3233 байт, збігаються = True
RAW успішно завантажено.

Перевірка повторного запуску:
Кількість після першого запуску: 6
Кількість після другого запуску: 6
Кількість не змінилася: True


In [68]:
def read_raw(table):
    raw_path = (
        f"{STUDENT}/raw/{table}/"
        f"load_date={LOAD_DATE}/{table}.json"
    )

    blob = lake.get_blob_client(raw_path)
    content = blob.download_blob().readall()

    return pd.read_json(
        BytesIO(content),
        dtype=False,
        convert_dates=False
    )

In [69]:
stations_raw = read_raw("stations")
sensors_raw = read_raw("sensors")
readings_raw = read_raw("readings")

print("Кількість рядків:")
print("stations_raw:", len(stations_raw))
print("sensors_raw:", len(sensors_raw))
print("readings_raw:", len(readings_raw))

Кількість рядків:
stations_raw: 7
sensors_raw: 8
readings_raw: 23


C:\Users\i.yarmolenko\AppData\Local\Temp\ipykernel_23184\627333964.py:10: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  return pd.read_json(
C:\Users\i.yarmolenko\AppData\Local\Temp\ipykernel_23184\627333964.py:10: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  return pd.read_json(
C:\Users\i.yarmolenko\AppData\Local\Temp\ipykernel_23184\627333964.py:10: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  return pd.read_json(


In [70]:
print("Міста (city):")
print(stations_raw["city"].unique())

print("\nКраїни (country):")
print(stations_raw["country"].unique())

print("\nПоказники (metric):")
print(sensors_raw["metric"].unique())

print("\nОдиниці виміру (unit):")
print(sensors_raw["unit"].unique())

print("\nСтатуси (status):")
print(readings_raw["status"].unique())

Міста (city):
<ArrowStringArray>
['Київ', 'Kyiv', 'Львів', 'Kraków', 'Warsaw', 'KYIV']
Length: 6, dtype: str

Країни (country):
<ArrowStringArray>
['UA', 'Ukraine', 'україна', 'PL', 'Poland']
Length: 5, dtype: str

Показники (metric):
<ArrowStringArray>
['PM2.5', 'CO2', 'pm2_5', 'PM 2.5', 'temperature', 'PM10', 'humidity']
Length: 7, dtype: str

Одиниці виміру (unit):
<ArrowStringArray>
['µg/m3', 'ppm', 'ug/m3', '°C', '%', nan]
Length: 6, dtype: str

Статуси (status):
<ArrowStringArray>
['ok', 'OK', 'error', 'calibrating']
Length: 4, dtype: str


In [71]:
# Копіюємо вихідну таблицю
stations_clean = stations_raw.copy()

# 1. Нормалізація назв міст
stations_clean["city"] = (
    stations_clean["city"]
    .str.strip()
    .str.lower()
    .map({
        "kyiv": "Kyiv",
        "київ": "Kyiv",
        "львів": "Lviv",
        "lviv": "Lviv",
        "kraków": "Kraków",
        "warsaw": "Warsaw"
    })
)

# 2. Нормалізація країн
stations_clean["country"] = (
    stations_clean["country"]
    .str.strip()
    .str.lower()
    .map({
        "ua": "UA",
        "ukraine": "UA",
        "україна": "UA",
        "pl": "PL",
        "poland": "PL"
    })
)

# 3. Перетворення installed_at на дату
iso = pd.to_datetime(
    stations_clean["installed_at"],
    format="ISO8601",
    errors="coerce",
    utc=True
)

dmy = pd.to_datetime(
    stations_clean["installed_at"],
    format="%d.%m.%Y",
    errors="coerce",
    utc=True
)

parsed = iso.fillna(dmy)
stations_clean["installed_at"] = parsed.dt.date

# 4. Видалення дублікатів за station_id
stations_clean = stations_clean.drop_duplicates(
    subset=["station_id"],
    keep="first"
)

# 5. Залишаємо потрібні колонки
stations_clean = stations_clean[
    ["station_id", "name", "city", "country", "installed_at"]
].copy()

# 6. Перевірка унікальних значень
print("Унікальні міста:")
print(stations_clean["city"].unique())

print("\nУнікальні країни:")
print(stations_clean["country"].unique())

print("\nКількість станцій:", len(stations_clean))

display(stations_clean)

Унікальні міста:
<ArrowStringArray>
['Kyiv', 'Lviv', 'Kraków', 'Warsaw']
Length: 4, dtype: str

Унікальні країни:
<ArrowStringArray>
['UA', 'PL']
Length: 2, dtype: str

Кількість станцій: 6


,station_id,name,city,country,installed_at
0,ST01,Центр,Kyiv,UA,2024-05-10
1,ST02,Оболонь,Kyiv,UA,2024-06-01
2,ST03,Личаків,Lviv,UA,2024-06-15
3,ST04,Stare Miasto,Kraków,PL,NaT
4,ST05,Centrum,Warsaw,PL,NaT
5,ST06,Позняки,Kyiv,UA,2024-08-20


In [72]:
# Копіюємо вихідну таблицю
sensors_clean = sensors_raw.copy()

# 1. Нормалізація назв метрик
sensors_clean["metric"] = (
    sensors_clean["metric"]
    .str.strip()
    .str.lower()
    .replace({
        "pm2_5": "PM2.5",
        "pm 2.5": "PM2.5",
        "pm2.5": "PM2.5",
        "pm10": "PM10",
        "co2": "CO2",
        "temperature": "temperature",
        "humidity": "humidity"
    })
)

# 2. Нормалізація одиниць виміру
sensors_clean["unit"] = (
    sensors_clean["unit"]
    .str.strip()
    .replace({
        "ug/m3": "µg/m3",
        "µg/m3": "µg/m3"
    })
)

# 3. Заповнення порожніх одиниць за метрикою
unit_by_metric = {
    "PM2.5": "µg/m3",
    "PM10": "µg/m3",
    "CO2": "ppm",
    "temperature": "°C",
    "humidity": "%"
}

sensors_clean["unit"] = sensors_clean["unit"].fillna(
    sensors_clean["metric"].map(unit_by_metric)
)

# 4. Залишаємо сенсори тільки наявних станцій
sensors_clean = sensors_clean[
    sensors_clean["station_id"].isin(stations_clean["station_id"])
].copy()

# 5. Залишаємо потрібні колонки
sensors_clean = sensors_clean[
    ["sensor_id", "station_id", "metric", "unit"]
].copy()

# 6. Перевірка одиниць для кожної метрики
print("Унікальні одиниці виміру за метриками:")

for metric, units in sensors_clean.groupby("metric")["unit"].unique().items():
    print(f"{metric}: {list(units)}")

print("\nКількість сенсорів:", len(sensors_clean))

display(sensors_clean)

Унікальні одиниці виміру за метриками:
CO2: ['ppm']
PM10: ['µg/m3']
PM2.5: ['µg/m3']
humidity: ['%']
temperature: ['°C']

Кількість сенсорів: 8


,sensor_id,station_id,metric,unit
0,SN100,ST01,PM2.5,µg/m3
1,SN101,ST01,CO2,ppm
2,SN102,ST02,PM2.5,µg/m3
3,SN103,ST03,PM2.5,µg/m3
4,SN104,ST03,temperature,°C
5,SN105,ST04,PM10,µg/m3
6,SN106,ST05,humidity,%
7,SN107,ST06,PM2.5,µg/m3


In [73]:
# Копіюємо вихідну таблицю
readings_work = readings_raw.copy()

# 1. Видаляємо повні дублікати рядків
readings_work = readings_work.drop_duplicates().copy()

# 2. Перетворюємо value на float з урахуванням десяткової коми
readings_work["value"] = pd.to_numeric(
    readings_work["value"].astype(str).str.replace(
        ",", ".", regex=False
    ),
    errors="coerce"
)

# 3. Зводимо status до нижнього регістру
readings_work["status"] = (
    readings_work["status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# 4. Видаляємо порожні та від'ємні value
readings_work = readings_work[
    readings_work["value"].notna()
    & (readings_work["value"] >= 0)
].copy()

# 5. Залишаємо тільки status = ok
readings_work = readings_work[
    readings_work["status"] == "ok"
].copy()

# 6. Залишаємо покази тільки від відомих сенсорів
readings_work = readings_work[
    readings_work["sensor_id"].isin(sensors_clean["sensor_id"])
].copy()

# 7. Перетворюємо measured_at на дату й час
measured_at = readings_work["measured_at"].astype("string").str.strip()

parsed_iso = pd.to_datetime(
    measured_at,
    format="ISO8601",
    errors="coerce"
)

parsed_dmy = pd.to_datetime(
    measured_at,
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

parsed_dot = pd.to_datetime(
    measured_at,
    format="%Y.%m.%d %H:%M",
    errors="coerce"
)

readings_work["measured_at"] = (
    parsed_iso.fillna(parsed_dmy).fillna(parsed_dot)
)

# Видаляємо порожні та некоректні дати
readings_work = readings_work[
    readings_work["measured_at"].notna()
].copy()

# 8. Додаємо дату виміру без часу
readings_work["measured_date"] = (
    readings_work["measured_at"].dt.date
)

# 9. Додаємо ознаку аномалії PM2.5 понад 500
readings_with_metric = readings_work.merge(
    sensors_clean[["sensor_id", "metric"]],
    on="sensor_id",
    how="left"
)

readings_with_metric["is_outlier"] = (
    (readings_with_metric["metric"] == "PM2.5")
    & (readings_with_metric["value"] > 500)
)

# 10. Залишаємо потрібні колонки
readings_clean = readings_with_metric[
    [
        "reading_id",
        "sensor_id",
        "value",
        "measured_at",
        "measured_date",
        "status",
        "is_outlier"
    ]
].copy()

# Перевірка результату
print("Кількість очищених показів:", len(readings_clean))
print("\nКількість аномальних показів:")
print(readings_clean["is_outlier"].sum())

print("\nПропущені значення:")
print(readings_clean.isna().sum())

display(readings_clean)

Кількість очищених показів: 18

Кількість аномальних показів:
1

Пропущені значення:
reading_id       0
sensor_id        0
value            0
measured_at      0
measured_date    0
status           0
is_outlier       0
dtype: int64


,reading_id,sensor_id,value,measured_at,measured_date,status,is_outlier
0,R0001,SN100,23.4,2026-07-01 08:00:00,2026-07-01,ok,False
1,R0002,SN100,31.7,2026-07-01 12:00:00,2026-07-01,ok,False
2,R0003,SN101,610.0,2026-07-01 08:00:00,2026-07-01,ok,False
3,R0004,SN102,18.9,2026-07-01 09:00:00,2026-07-01,ok,False
4,R0005,SN103,45.2,2026-07-01 10:00:00,2026-07-01,ok,False
5,R0006,SN104,27.5,2026-07-01 10:00:00,2026-07-01,ok,False
6,R0008,SN105,52.0,2026-07-02 09:00:00,2026-07-02,ok,False
7,R0009,SN106,64.0,2026-07-02 09:00:00,2026-07-02,ok,False
8,R0011,SN102,22.1,2026-07-02 12:00:00,2026-07-02,ok,False
9,R0012,SN100,5000.0,2026-07-03 08:00:00,2026-07-03,ok,True


In [74]:
# Словник для збереження результатів
quality_rows = []

# 1. STATIONS — дублікати station_id
before = len(stations_raw)

stations_check = stations_raw.drop_duplicates(
    subset=["station_id"],
    keep="first"
).copy()

quality_rows.append({
    "table": "stations",
    "rule": "duplicate_station_id",
    "rows_affected": before - len(stations_check)
})


# 2. SENSORS — невідомі station_id
before = len(sensors_raw)

sensors_check = sensors_raw[
    sensors_raw["station_id"].isin(stations_clean["station_id"])
].copy()

quality_rows.append({
    "table": "sensors",
    "rule": "unknown_station_id",
    "rows_affected": before - len(sensors_check)
})


# 3. READINGS — повні дублікати рядків
readings_check = readings_raw.copy()

before = len(readings_check)
readings_check = readings_check.drop_duplicates().copy()

quality_rows.append({
    "table": "readings",
    "rule": "duplicate_row",
    "rows_affected": before - len(readings_check)
})


# 4. READINGS — порожні або від'ємні value
before = len(readings_check)

readings_check["value"] = pd.to_numeric(
    readings_check["value"].astype(str).str.replace(
        ",", ".", regex=False
    ),
    errors="coerce"
)

readings_check = readings_check[
    readings_check["value"].notna()
    & (readings_check["value"] >= 0)
].copy()

quality_rows.append({
    "table": "readings",
    "rule": "invalid_value",
    "rows_affected": before - len(readings_check)
})


# 5. READINGS — статус, відмінний від ok
before = len(readings_check)

readings_check["status"] = (
    readings_check["status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

readings_check = readings_check[
    readings_check["status"] == "ok"
].copy()

quality_rows.append({
    "table": "readings",
    "rule": "status_not_ok",
    "rows_affected": before - len(readings_check)
})


# 6. READINGS — невідомий sensor_id
before = len(readings_check)

readings_check = readings_check[
    readings_check["sensor_id"].isin(sensors_clean["sensor_id"])
].copy()

quality_rows.append({
    "table": "readings",
    "rule": "unknown_sensor_id",
    "rows_affected": before - len(readings_check)
})


# 7. READINGS — порожній або некоректний measured_at
before = len(readings_check)

measured_at = (
    readings_check["measured_at"]
    .astype("string")
    .str.strip()
)

parsed_iso = pd.to_datetime(
    measured_at,
    format="ISO8601",
    errors="coerce"
)

parsed_dmy = pd.to_datetime(
    measured_at,
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

parsed_dot = pd.to_datetime(
    measured_at,
    format="%Y.%m.%d %H:%M",
    errors="coerce"
)

readings_check["measured_at"] = (
    parsed_iso.fillna(parsed_dmy).fillna(parsed_dot)
)

readings_check = readings_check[
    readings_check["measured_at"].notna()
].copy()

quality_rows.append({
    "table": "readings",
    "rule": "invalid_measured_at",
    "rows_affected": before - len(readings_check)
})


# 8. READINGS — аномальні значення PM2.5
readings_check = readings_check.merge(
    sensors_clean[["sensor_id", "metric"]],
    on="sensor_id",
    how="left"
)

readings_check["is_outlier"] = (
    (readings_check["metric"] == "PM2.5")
    & (readings_check["value"] > 500)
)

quality_rows.append({
    "table": "readings",
    "rule": "pm25_outlier",
    "rows_affected": int(readings_check["is_outlier"].sum())
})


# Формуємо звіт
quality_report = pd.DataFrame(
    quality_rows,
    columns=["table", "rule", "rows_affected"]
)

display(quality_report)

,table,rule,rows_affected
0,stations,duplicate_station_id,1
1,sensors,unknown_station_id,0
2,readings,duplicate_row,1
3,readings,invalid_value,2
4,readings,status_not_ok,0
5,readings,unknown_sensor_id,1
6,readings,invalid_measured_at,1
7,readings,pm25_outlier,1


In [75]:
# 1. Виключаємо аномальні покази
readings_for_mart = readings_clean[
    readings_clean["is_outlier"] == False
].copy()

# 2. Об'єднуємо покази із сенсорами
mart_data = readings_for_mart.merge(
    sensors_clean[["sensor_id", "station_id", "metric"]],
    on="sensor_id",
    how="inner"
)

# 3. Додаємо місто зі станцій
mart_data = mart_data.merge(
    stations_clean[["station_id", "city"]],
    on="station_id",
    how="inner"
)

# 4. Групуємо за містом і метрикою
avg_by_city_metric = (
    mart_data
    .groupby(["city", "metric"], as_index=False)
    .agg(
        avg_value=("value", "mean"),
        readings_count=("value", "count")
    )
)

# 5. Округлюємо середнє до двох знаків
avg_by_city_metric["avg_value"] = (
    avg_by_city_metric["avg_value"].round(2)
)

# 6. Сортуємо за містом і метрикою
avg_by_city_metric = avg_by_city_metric.sort_values(
    ["city", "metric"]
).reset_index(drop=True)

# 7. Переглядаємо результат
display(avg_by_city_metric)

# 8. Записуємо вітрину у Parquet
mart_path = (
    f"{STUDENT}/mart/avg_by_city_metric/"
    "avg_by_city_metric.parquet"
)

parquet_data = avg_by_city_metric.to_parquet(index=False)

lake.get_blob_client(mart_path).upload_blob(
    parquet_data,
    overwrite=True
)

print("Вітрину успішно збережено:", mart_path)

,city,metric,avg_value,readings_count
0,Kraków,PM10,52.5,3
1,Kyiv,CO2,665.0,2
2,Kyiv,PM2.5,23.8,6
3,Lviv,PM2.5,41.6,3
4,Lviv,temperature,27.5,1
5,Warsaw,humidity,67.5,2


Вітрину успішно збережено: yarmolenko_iryna/mart/avg_by_city_metric/avg_by_city_metric.parquet


In [76]:
# 1. Виключаємо аномальні покази
readings_for_mart = readings_clean[
    readings_clean["is_outlier"] == False
].copy()

# 2. Об'єднуємо покази із сенсорами
pm25_data = readings_for_mart.merge(
    sensors_clean[["sensor_id", "station_id", "metric", "unit"]],
    on="sensor_id",
    how="inner"
)

# 3. Відбираємо PM2.5 понад 25 µg/m3
pm25_exceedances_data = pm25_data[
    (pm25_data["metric"] == "PM2.5")
    & (pm25_data["unit"] == "µg/m3")
    & (pm25_data["value"] > 25)
].copy()

# 4. Додаємо назву станції та місто
pm25_exceedances_data = pm25_exceedances_data.merge(
    stations_clean[["station_id", "name", "city"]],
    on="station_id",
    how="inner"
)

# 5. Рахуємо кількість перевищень за станціями
pm25_exceedances = (
    pm25_exceedances_data
    .groupby(["station_id", "name", "city"], as_index=False)
    .agg(
        exceedances_count=("value", "count")
    )
    .rename(columns={"name": "station_name"})
)

# 6. Сортуємо за кількістю перевищень і station_id
pm25_exceedances = pm25_exceedances.sort_values(
    ["exceedances_count", "station_id"],
    ascending=[False, True]
).reset_index(drop=True)

# 7. Виводимо результат
display(pm25_exceedances)

# 8. Зберігаємо в Parquet
mart_path = (
    f"{STUDENT}/mart/pm25_exceedances/"
    "pm25_exceedances.parquet"
)

parquet_data = pm25_exceedances.to_parquet(index=False)

lake.get_blob_client(mart_path).upload_blob(
    parquet_data,
    overwrite=True
)

print("Вітрину успішно збережено:", mart_path)

,station_id,station_name,city,exceedances_count
0,ST03,Личаків,Lviv,3
1,ST01,Центр,Kyiv,2


Вітрину успішно збережено: yarmolenko_iryna/mart/pm25_exceedances/pm25_exceedances.parquet


In [77]:
# 1. Виключаємо аномальні покази
readings_for_mart = readings_clean[
    readings_clean["is_outlier"] == False
].copy()

# 2. Додаємо метрику із таблиці sensors_clean
daily_data = readings_for_mart.merge(
    sensors_clean[["sensor_id", "metric"]],
    on="sensor_id",
    how="inner"
)

# 3. Рахуємо середнє значення за датою і метрикою
daily_trend = (
    daily_data
    .groupby(["measured_date", "metric"], as_index=False)
    .agg(
        avg_value=("value", "mean")
    )
)

# 4. Округлюємо середнє до двох знаків
daily_trend["avg_value"] = daily_trend["avg_value"].round(2)

# 5. Сортуємо за датою і метрикою
daily_trend = daily_trend.sort_values(
    ["measured_date", "metric"]
).reset_index(drop=True)

# 6. Переглядаємо результат
display(daily_trend)

# 7. Записуємо окремий Parquet-файл для кожного місяця
for (year, month), part in daily_trend.groupby(
    [
        daily_trend["measured_date"].apply(lambda d: d.year),
        daily_trend["measured_date"].apply(lambda d: d.month)
    ]
):
    mart_path = (
        f"{STUDENT}/mart/daily_trend/"
        f"year={year}/month={month:02d}/part.parquet"
    )

    parquet_data = part[
        ["measured_date", "metric", "avg_value"]
    ].to_parquet(index=False)

    lake.get_blob_client(mart_path).upload_blob(
        parquet_data,
        overwrite=True
    )

    print("Збережено:", mart_path)

print("Вітрину daily_trend успішно збережено.")

,measured_date,metric,avg_value
0,2026-07-01,CO2,610.00
1,2026-07-01,PM2.5,29.80
2,2026-07-01,temperature,27.50
3,2026-07-02,PM10,52.00
4,2026-07-02,PM2.5,22.10
5,2026-07-02,humidity,64.00
6,2026-07-03,PM10,49.80
7,2026-07-03,PM2.5,38.60
8,2026-07-04,CO2,720.00
9,2026-07-04,PM2.5,29.23


Збережено: yarmolenko_iryna/mart/daily_trend/year=2026/month=07/part.parquet
Вітрину daily_trend успішно збережено.


In [78]:
# 1. Виводимо всі блоби власного префікса
prefix = f"{STUDENT}/"

print("Блоби у сховищі:")

for blob in lake.list_blobs(name_starts_with=prefix):
    print(f"{blob.name} — {blob.size} байт")


# 2. Читаємо вітрину pm25_exceedances з Parquet
mart_path = (
    f"{STUDENT}/mart/pm25_exceedances/"
    "pm25_exceedances.parquet"
)

blob = lake.get_blob_client(mart_path)

parquet_data = blob.download_blob().readall()

pm25_check = pd.read_parquet(
    BytesIO(parquet_data),
    columns=["station_name", "exceedances_count"]
)

# 3. Виводимо результат
print("\nПеревірка вітрини pm25_exceedances:")
display(pm25_check)

Блоби у сховищі:
yarmolenko_iryna/mart/2026-11-04/air_quality_mart.parquet — 7522 байт
yarmolenko_iryna/mart/avg_by_city_metric/avg_by_city_metric.parquet — 2956 байт
yarmolenko_iryna/mart/daily_trend/year=2026/month=07/part.parquet — 2438 байт
yarmolenko_iryna/mart/pm25_exceedances/pm25_exceedances.parquet — 2903 байт
yarmolenko_iryna/raw/2026-11-04/readings.json — 3233 байт
yarmolenko_iryna/raw/2026-11-04/sensors.json — 833 байт
yarmolenko_iryna/raw/2026-11-04/stations.json — 996 байт
yarmolenko_iryna/raw/readings/load_date=2026-11-04/readings.json — 3233 байт
yarmolenko_iryna/raw/sensors/load_date=2026-11-04/sensors.json — 833 байт
yarmolenko_iryna/raw/stations/load_date=2026-11-04/stations.json — 996 байт
yarmolenko_iryna/test_permission.txt — 15 байт

Перевірка вітрини pm25_exceedances:


,station_name,exceedances_count
0,Личаків,3
1,Центр,2
